In [3]:
# all imports

import json, re, time
from collections import Counter
from pathlib import Path
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

In [4]:
# define paths

BASE     = Path("../dataset")
ADAPTER  = Path("../adapters/lora_v1/epoch_3")
OUT_DIR  = Path("../outputs/test_final")
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

In [3]:
MAX_NEW_TOKENS = 563          
RETRY_PENALTY  = 1.3

OVERWRITE = False             # protects saved predictions from accidental overwrite

assert ADAPTER.exists(), f"Adapter not found: {ADAPTER}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if (OUT_DIR / "predictions.jsonl").exists() and not OVERWRITE:
    raise RuntimeError("Predictions already exist. Delete the folder or set OVERWRITE = True.")
print("Paths OK")

Paths OK


In [6]:
# load the base model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="cuda", torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [5]:
# load epoch-3 adapter

model = PeftModel.from_pretrained(model, ADAPTER)   # attaches the trained LoRA weights
model.eval()
print(f"Adapter loaded: {ADAPTER}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Adapter loaded: ../adapters/lora_v1/epoch_3
GPU memory: 5.68 GB


In [6]:
# load the test sequence

with open(BASE / "sequences.json") as f:
    sequences = json.load(f)
by_id = {s["sequence_id"]: s for s in sequences if s["split"] == "test"}

with open(BASE / "sample_submission.jsonl") as f:
    sample_ids = [json.loads(l)["sequence_id"] for l in f]

assert set(sample_ids) == set(by_id), "Test IDs in sequences.json and sample_submission differ"
test_seqs = [by_id[i] for i in sample_ids]
print(len(test_seqs), "test sequences")

15 test sequences


In [ ]:
# prompt same as training

PAGE_PROMPT = (
    "This is page {n} of 3 of an English manga sequence. "
    "Extract the story text of this page in reading order and say who speaks each line. "
    "Label characters character1, character2, ... Use UNKNOWN if the speaker is unclear "
    "and NARRATION for narrator boxes. "
    "Exclude sound effects, titles, credits, page numbers, and text on signs or documents. "
    'Output a JSON list only: [{{"speaker":...,"text":...}}]. Use [] if the page has no story text.'
)

def context_block(prev_pages):
    if not prev_pages:
        return ""
    lines = [f"Page {i} result: " + json.dumps(p, ensure_ascii=False, separators=(",", ":"))
             for i, p in enumerate(prev_pages, 1)]
    return ("Earlier pages of this sequence (reuse the same labels for the same characters):\n"
            + "\n".join(lines) + "\n")

def load_image(rel_path):
    return Image.open(BASE / rel_path).convert("RGB")

def make_messages(image, page_idx, prev_pages):
    text = context_block(prev_pages) + PAGE_PROMPT.format(n=page_idx + 1)
    return [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": text},
    ]}]

def parse_page(raw):
    out = []
    for m in re.finditer(r'\{[^{}]*\}', raw):
        try:
            o = json.loads(m.group())
        except json.JSONDecodeError:
            continue
        if not (isinstance(o, dict) and "speaker" in o and "text" in o):
            continue
        speaker, text = str(o["speaker"]).strip(), str(o["text"]).strip()
        if not text:
            continue
        out.append({"speaker": speaker, "text": text})
    return out

def collapse_runs(items, max_run=2):
    out, run = [], 0
    for it in items:
        if out and it == out[-1]:
            run += 1
            if run >= max_run:
                continue
        else:
            run = 0
        out.append(it)
    return out

def looks_looped(raw, parsed):
    truncated = not raw.strip().endswith("]")
    repeated = max(Counter((i["speaker"], i["text"]) for i in parsed).values(), default=0) >= 3
    return truncated or repeated

@torch.no_grad()
def generate_page(rel_path, page_idx, prev_pages, repetition_penalty=1.0):
    image = load_image(rel_path)
    prompt_text = processor.apply_chat_template(
        make_messages(image, page_idx, prev_pages), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to("cuda")
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         use_cache=True, repetition_penalty=repetition_penalty)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def predict_page_auto(rel_path, page_idx, prev_pages):
    raw = generate_page(rel_path, page_idx, prev_pages)
    parsed = collapse_runs(parse_page(raw))
    retried = False
    if looks_looped(raw, parsed):
        retried = True
        raw2 = generate_page(rel_path, page_idx, prev_pages, repetition_penalty=RETRY_PENALTY)
        parsed2 = collapse_runs(parse_page(raw2))
        if not looks_looped(raw2, parsed2):
            raw, parsed = raw2, parsed2
        else:
            parsed = parsed2 if len(parsed2) <= len(parsed) else parsed
    return raw, parsed

print("Functions defined!")

Functions defined!


In [8]:
# load test sequences

with open(BASE / "sequences.json") as f:
    sequences = json.load(f)
by_id = {s["sequence_id"]: s for s in sequences if s["split"] == "test"}

with open(BASE / "sample_submission.jsonl") as f:
    sample_ids = [json.loads(l)["sequence_id"] for l in f]

assert set(sample_ids) == set(by_id), "Test IDs differ between sequences.json and sample_submission"
test_seqs = [by_id[i] for i in sample_ids]
print(len(test_seqs), "test sequences. No labels are read in this notebook.")

15 test sequences. No labels are read in this notebook.


In [ ]:
start = time.time()
with open(OUT_DIR / "predictions.jsonl", "w") as pf, open(OUT_DIR / "raw.jsonl", "w") as rf:
    for n, seq in enumerate(test_seqs, 1):
        pages, raws = [], []
        for k, rel in enumerate(seq["images"]):
            raw, parsed = predict_page_auto(rel, k, pages)
            raws.append(raw); pages.append(parsed)
        pf.write(json.dumps({"sequence_id": seq["sequence_id"], "pages": pages}) + "\n")
        rf.write(json.dumps({"sequence_id": seq["sequence_id"], "raw": raws}) + "\n")
        pf.flush(); rf.flush()
        print(f"[{n}/{len(test_seqs)}] {seq['sequence_id']} ({time.time()-start:.0f}s) "
              f"lines: {[len(p) for p in pages]}")
print("Inference done.")

[1/15] seq_6b18300406fa878f (28s) lines: [0, 11, 8]
[2/15] seq_c780ac6d1dbd1242 (56s) lines: [10, 3, 12]
[3/15] seq_5303b549f62e9d3f (75s) lines: [7, 7, 1]
[4/15] seq_d455cfd18edfb0ba (96s) lines: [10, 8, 4]
[5/15] seq_e04ef3d20fd0baa8 (104s) lines: [3, 1, 3]
[6/15] seq_6aa313a0c0fa1c39 (111s) lines: [2, 3, 2]
[7/15] seq_32c47ca11b40ca8a (144s) lines: [2, 1, 4]
[8/15] seq_81b1782990ab7950 (190s) lines: [7, 16, 18]
[9/15] seq_14469aeb38963085 (235s) lines: [14, 12, 12]
[10/15] seq_e9ffbb64217c2fed (274s) lines: [11, 13, 7]
[11/15] seq_316bc7bd43ed6996 (323s) lines: [19, 26, 5]
[12/15] seq_cf7df70d4bba5da5 (358s) lines: [13, 13, 2]
[13/15] seq_e456a0394af99570 (376s) lines: [6, 3, 3]
[14/15] seq_c5622aca7487a661 (389s) lines: [1, 4, 4]
[15/15] seq_75aeca6c7131544b (401s) lines: [1, 5, 4]
Inference done.


In [10]:
# format check 

preds = [json.loads(l) for l in open(OUT_DIR / "predictions.jsonl")]

assert len(preds) == len(sample_ids), "Wrong number of sequences"
assert [p["sequence_id"] for p in preds] == sample_ids, "IDs or order differ from sample_submission"

problems = []
for p in preds:
    if set(p) != {"sequence_id", "pages"}:
        problems.append((p["sequence_id"], "wrong top-level keys"))
    if len(p["pages"]) != 3:
        problems.append((p["sequence_id"], "not 3 pages"))
        continue
    for k, page in enumerate(p["pages"], 1):
        if not isinstance(page, list):
            problems.append((p["sequence_id"], k, "page is not a list")); continue
        for item in page:
            if set(item) != {"speaker", "text"}:
                problems.append((p["sequence_id"], k, "wrong keys", item))
            elif not isinstance(item["speaker"], str) or not item["speaker"].strip():
                problems.append((p["sequence_id"], k, "bad speaker", item))
            elif not isinstance(item["text"], str) or not item["text"].strip():
                problems.append((p["sequence_id"], k, "empty text", item))

if problems:
    for x in problems:
        print("PROBLEM:", x)
    raise SystemExit(f"{len(problems)} format problem(s). Submission NOT written.")

with open(OUT_DIR / "submission.jsonl", "w") as f:
    for p in preds:
        f.write(json.dumps({"sequence_id": p["sequence_id"], "pages": p["pages"]}) + "\n")
print("All format checks passed:", len(preds), "sequences ->", OUT_DIR / "submission.jsonl")

All format checks passed: 15 sequences -> ../outputs/test_final/submission.jsonl


In [ ]:
# sanity report

raws = [json.loads(l) for l in open(OUT_DIR / "raw.jsonl")]

print("Lines per page (p1, p2, p3):")
for p in preds:
    print(" ", p["sequence_id"], [len(pg) for pg in p["pages"]])

print(f"\nEmpty pages: {sum(1 for p in preds for pg in p['pages'] if not pg)} of {len(preds)*3}")
print("Speaker labels:", dict(Counter(i["speaker"] for p in preds for pg in p["pages"] for i in pg).most_common(12)))

trunc = [(r["sequence_id"], k + 1) for r in raws for k, t in enumerate(r["raw"]) if not t.strip().endswith("]")]
rep = [(p["sequence_id"], k + 1) for p in preds for k, pg in enumerate(p["pages"])
       if max(Counter((i["speaker"], i["text"]) for i in pg).values(), default=0) >= 3]
long_pages = [(p["sequence_id"], k + 1, len(pg)) for p in preds for k, pg in enumerate(p["pages"]) if len(pg) > 20]
print("Raw output not ending in ']':", trunc or "none")
print("Same line 3+ times on a page:", rep or "none")
print("Pages with more than 20 lines:", long_pages or "none")

Lines per page (p1, p2, p3):
  seq_6b18300406fa878f [0, 11, 8]
  seq_c780ac6d1dbd1242 [10, 3, 12]
  seq_5303b549f62e9d3f [7, 7, 1]
  seq_d455cfd18edfb0ba [10, 8, 4]
  seq_e04ef3d20fd0baa8 [3, 1, 3]
  seq_6aa313a0c0fa1c39 [2, 3, 2]
  seq_32c47ca11b40ca8a [2, 1, 4]
  seq_81b1782990ab7950 [7, 16, 18]
  seq_14469aeb38963085 [14, 12, 12]
  seq_e9ffbb64217c2fed [11, 13, 7]
  seq_316bc7bd43ed6996 [19, 26, 5]
  seq_cf7df70d4bba5da5 [13, 13, 2]
  seq_e456a0394af99570 [6, 3, 3]
  seq_c5622aca7487a661 [1, 4, 4]
  seq_75aeca6c7131544b [1, 5, 4]

Empty pages: 1 of 45
Speaker labels: {'character1': 109, 'character2': 96, 'character3': 57, 'character5': 23, 'character6': 22, 'NARRATION': 9, 'character4': 4, 'UNKNOWN': 1}
Raw output not ending in ']': none
Same line 3+ times on a page: none
Pages with more than 20 lines: [('seq_316bc7bd43ed6996', 2, 26)]
